# L40S 실습: FEM 솔버 → 서로게이트 모델 학습

**목적**

자동차 CAE 업무 맥락에서, KADaP GPU 서버 존의 **L40S** 자원이 어떤 역할에 적합한지를 직접 손으로 확인하는 실습입니다.

앞선 스터디에서 정리한 구조를 그대로 재현합니다.

| 스터디에서 정리한 개념 | 이 노트북에서의 대응 |
|---|---|
| PhysX = 물리 솔버 (판단 없음) | 2D 평면응력 FEM 솔버 |
| DGX에서의 학습 = 서로게이트 피팅 | L40S에서 신경망 서로게이트 학습 |
| Jetson 배포 = 실시간 추론 | 서로게이트 단일 케이스 추론 지연 측정 |

**L40S 관점의 핵심 포인트**

L40S는 FP32/TF32 연산은 강하지만 **FP64(배정밀도)는 약합니다.** 전통적 FEM·CFD 솔버는 FP64를 쓰는 경우가 많아 솔버 자체를 GPU로 옮기는 데는 A100이 유리합니다. 반대로 **서로게이트 모델 학습은 FP32/TF32로 충분**하므로 L40S가 적합합니다. 6번 항목에서 이 특성을 직접 측정해 확인합니다.

**진행 순서**

1. 환경 확인 (GPU / VRAM)
2. FEM 솔버 구현 및 보 이론 대조 검증
3. 파라메트릭 해석 데이터셋 생성 (설계변수 3개 → 응답 2개)
4. 서로게이트 모델 학습 (L40S)
5. 정확도 검증 및 속도 비교 (FEM 대비 몇 배 빠른가)
6. FP32 vs FP64 GPU 벤치마크 (L40S 특성 확인)
7. 결과 저장 (MyDisk)

> 워크로드 생성 시 GPU는 **L40S**를 선택하고, 이미지는 PyTorch+CUDA Built-in 이미지를 사용하세요.

## 1. 환경 확인

In [ ]:
import torch, subprocess, platform

print("Python :", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA   :", torch.cuda.is_available())

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {p.name} | VRAM {p.total_memory/1024**3:.1f} GB | SM {p.major}.{p.minor}")
    GPU_NAME = torch.cuda.get_device_properties(0).name
else:
    GPU_NAME = "CPU only"
    print("  GPU 미할당 - 워크로드의 GPU 노드 설정을 확인하세요.")

print()
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total,driver_version",
                      "--format=csv"], capture_output=True, text=True).stdout)

## 2. FEM 솔버 구현 및 검증

외팔보(cantilever)를 2D 평면응력 Q4 요소로 이산화해 푸는 최소 FEM 솔버입니다.
`numpy` 만 사용하며, CPU에서 FP64로 계산합니다 (일반적인 상용 CAE 솔버와 같은 정밀도).

- 입력(설계변수): 단면 높이 `H`, 두께 `t`, 하중 `P`
- 출력(응답): 최대 von Mises 응력, 자유단 처짐

In [ ]:
import numpy as np

def solve_cantilever(L=1.0, H=0.1, t=0.01, E=210e9, nu=0.3, P=1000.0, nx=40, ny=8):
    """2D 평면응력 FEM (Q4, 2x2 Gauss). x=0 고정, x=L 자유단에 하중 P(하방).
    반환: (최대 von Mises 응력 [Pa], 자유단 처짐 [m])"""
    dx, dy = L/nx, H/ny
    nnx, nny = nx+1, ny+1
    nid = lambda i, j: j*nnx + i
    ndof = nnx*nny*2

    # 평면응력 구성행렬
    D = E/(1-nu**2)*np.array([[1, nu, 0], [nu, 1, 0], [0, 0, (1-nu)/2]])

    # Q4 요소강성 (2x2 Gauss 적분)
    g = 1/np.sqrt(3)
    ke = np.zeros((8, 8)); Bs = []
    for xi, eta in [(-g,-g), (g,-g), (g,g), (-g,g)]:
        dN = np.array([[-(1-eta), (1-eta), (1+eta), -(1+eta)],
                       [-(1-xi), -(1+xi), (1+xi),  (1-xi)]])*0.25
        dNxy = np.linalg.solve(np.array([[dx/2, 0], [0, dy/2]]), dN)
        B = np.zeros((3, 8))
        B[0, 0::2] = dNxy[0]; B[1, 1::2] = dNxy[1]
        B[2, 0::2] = dNxy[1]; B[2, 1::2] = dNxy[0]
        ke += B.T @ D @ B * (dx*dy/4) * t
        Bs.append(B)

    # 전역강성 조립
    K = np.zeros((ndof, ndof)); elems = []
    for j in range(ny):
        for i in range(nx):
            n = [nid(i,j), nid(i+1,j), nid(i+1,j+1), nid(i,j+1)]
            dofs = np.array([[2*k, 2*k+1] for k in n]).ravel()
            elems.append((n, dofs))
            K[np.ix_(dofs, dofs)] += ke

    # 하중: 자유단 우측 절단면 절점에 분배
    F = np.zeros(ndof)
    right = [nid(nx, j) for j in range(nny)]
    F[[2*k+1 for k in right]] = -P/len(right)

    # 경계조건: x=0 완전고정
    fixed = [d for j in range(nny) for d in (2*nid(0,j), 2*nid(0,j)+1)]
    free = np.setdiff1d(np.arange(ndof), fixed)

    U = np.zeros(ndof)
    U[free] = np.linalg.solve(K[np.ix_(free, free)], F[free])

    # von Mises 최대값 (Gauss point 기준)
    vm_max = 0.0
    for n, dofs in elems:
        ue = U[dofs]
        for B in Bs:
            sx, sy, sxy = D @ (B @ ue)
            vm_max = max(vm_max, np.sqrt(sx**2 - sx*sy + sy**2 + 3*sxy**2))

    return vm_max, abs(U[2*nid(nx, ny//2)+1])

print("솔버 정의 완료")

### 2-1. 보 이론과 대조 검증

In [ ]:
import time

L, H, t, E, P = 1.0, 0.1, 0.01, 210e9, 1000.0

t0 = time.perf_counter()
vm_fem, tip_fem = solve_cantilever(L=L, H=H, t=t, E=E, P=P)
t_solve = time.perf_counter() - t0

I = t*H**3/12
tip_beam = P*L**3/(3*E*I)
sig_beam = P*L*(H/2)/I

print(f"{'항목':<22}{'FEM':>14}{'보 이론':>14}{'오차':>10}")
print("-"*60)
print(f"{'자유단 처짐 [mm]':<22}{tip_fem*1000:>14.4f}{tip_beam*1000:>14.4f}{abs(tip_fem-tip_beam)/tip_beam*100:>9.1f}%")
print(f"{'최대 응력 [MPa]':<22}{vm_fem/1e6:>14.2f}{sig_beam/1e6:>14.2f}{abs(vm_fem-sig_beam)/sig_beam*100:>9.1f}%")
print()
print(f"1회 해석 소요 시간: {t_solve*1000:.1f} ms")
print()
print("※ Q4 요소의 전단잠김(shear locking)으로 처짐이 다소 과소평가되고,")
print("   von Mises는 고정단 표면이 아닌 Gauss point에서 평가되어 응력이 낮게 나옵니다.")
print("   수 % 수준의 차이는 정상이며, 솔버가 물리적으로 타당하게 동작함을 의미합니다.")

## 3. 파라메트릭 해석 데이터셋 생성

설계변수 3개(`H`, `t`, `P`)를 무작위 샘플링해 FEM을 반복 실행합니다.
**이 단계가 CAE 실무에서 가장 비싼 부분** — 상용 솔버로 대규모 DOE를 돌리면 수 시간~수일이 걸리는 지점입니다.

In [ ]:
N_SAMPLES = 1000

rng = np.random.default_rng(42)
H_s = rng.uniform(0.05, 0.20, N_SAMPLES)    # 단면 높이 50~200 mm
t_s = rng.uniform(0.005, 0.03, N_SAMPLES)   # 두께 5~30 mm
P_s = rng.uniform(500, 5000, N_SAMPLES)     # 하중 0.5~5 kN

X_raw = np.stack([H_s, t_s, P_s], axis=1)
Y_raw = np.zeros((N_SAMPLES, 2))

t0 = time.perf_counter()
for k in range(N_SAMPLES):
    Y_raw[k] = solve_cantilever(L=L, H=H_s[k], t=t_s[k], E=E, P=P_s[k])
    if (k+1) % 200 == 0:
        el = time.perf_counter() - t0
        print(f"  {k+1:4d}/{N_SAMPLES} 완료 | 경과 {el:6.1f} s | 케이스당 {el/(k+1)*1000:5.1f} ms")

T_DATASET = time.perf_counter() - t0
T_PER_SOLVE = T_DATASET / N_SAMPLES

print()
print(f"데이터셋 생성 완료: {N_SAMPLES}건, 총 {T_DATASET:.1f} s")
print(f"FEM 1케이스 평균: {T_PER_SOLVE*1000:.2f} ms")
print()
print(f"응력 범위 : {Y_raw[:,0].min()/1e6:8.2f} ~ {Y_raw[:,0].max()/1e6:8.2f} MPa")
print(f"처짐 범위 : {Y_raw[:,1].min()*1000:8.4f} ~ {Y_raw[:,1].max()*1000:8.4f} mm")

## 4. 서로게이트 모델 학습 (L40S)

응답이 수 자릿수에 걸쳐 분포하므로 **로그 변환 후 표준화**해 학습합니다.
CAE 서로게이트에서 흔히 쓰는 전처리 방식입니다.

In [ ]:
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("학습 디바이스:", device, "|", GPU_NAME)

# TF32 허용 (Ampere/Ada 이상에서 FP32 행렬연산 가속)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# 전처리: 입력/출력 모두 로그 변환 후 표준화
Xl, Yl = np.log(X_raw), np.log(Y_raw)
x_mu, x_sd = Xl.mean(0), Xl.std(0)
y_mu, y_sd = Yl.mean(0), Yl.std(0)

# 학습/검증 분할 (80% / 20%)
n_tr = int(N_SAMPLES*0.8)
idx = rng.permutation(N_SAMPLES)
tr, va = idx[:n_tr], idx[n_tr:]
print(f"학습 {len(tr)}건 / 검증 {len(va)}건")

to_t = lambda a: torch.tensor(a, dtype=torch.float32, device=device)
Xtr, Ytr = to_t((Xl[tr]-x_mu)/x_sd), to_t((Yl[tr]-y_mu)/y_sd)
Xva, Yva = to_t((Xl[va]-x_mu)/x_sd), to_t((Yl[va]-y_mu)/y_sd)

torch.manual_seed(0)
model = nn.Sequential(
    nn.Linear(3, 128), nn.SiLU(),
    nn.Linear(128, 128), nn.SiLU(),
    nn.Linear(128, 64), nn.SiLU(),
    nn.Linear(64, 2),
).to(device)

print("파라미터 수:", sum(p.numel() for p in model.parameters()))

opt = torch.optim.Adam(model.parameters(), lr=2e-3)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=3000)
lossf = nn.MSELoss()

t0 = time.perf_counter()
for ep in range(3000):
    model.train(); opt.zero_grad()
    loss = lossf(model(Xtr), Ytr)
    loss.backward(); opt.step(); sched.step()
    if (ep+1) % 500 == 0:
        model.eval()
        with torch.no_grad():
            vl = lossf(model(Xva), Yva).item()
        print(f"  epoch {ep+1:4d} | train {loss.item():.3e} | valid {vl:.3e}")
if torch.cuda.is_available():
    torch.cuda.synchronize()
T_TRAIN = time.perf_counter() - t0
print()
print(f"학습 소요 시간: {T_TRAIN:.1f} s")

## 5. 정확도 검증 및 속도 비교

In [ ]:
model.eval()
with torch.no_grad():
    pred_n = model(Xva).cpu().numpy()

# 역변환: 표준화 해제 -> exp
pred = np.exp(pred_n*y_sd + y_mu)
true = Y_raw[va]

err = np.abs(pred-true)/true*100
names = ["최대 von Mises 응력", "자유단 처짐"]

print(f"=== 검증셋({len(va)}건) 예측 정확도 ===")
for i, nm in enumerate(names):
    print(f"{nm:<22} 평균오차 {err[:,i].mean():5.2f}%  |  최대오차 {err[:,i].max():5.2f}%  |  95%분위 {np.percentile(err[:,i],95):5.2f}%")

print()
print("샘플 5건 비교 (H[mm], t[mm], P[N] -> 응력[MPa], 처짐[mm])")
print(f"{'H':>6}{'t':>6}{'P':>7} | {'응력_FEM':>9}{'응력_대체':>9} | {'처짐_FEM':>9}{'처짐_대체':>9}")
print("-"*72)
for k in range(5):
    h, tt, pp = X_raw[va][k]
    print(f"{h*1000:6.1f}{tt*1000:6.1f}{pp:7.0f} | {true[k,0]/1e6:9.2f}{pred[k,0]/1e6:9.2f} | {true[k,1]*1000:9.4f}{pred[k,1]*1000:9.4f}")

In [ ]:
# 추론 지연 측정: 단일 케이스 (실시간 응답 시나리오)
x1 = Xva[:1]
with torch.no_grad():
    for _ in range(50): model(x1)              # 워밍업
    if torch.cuda.is_available(): torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(1000): model(x1)
    if torch.cuda.is_available(): torch.cuda.synchronize()
    T_INFER_1 = (time.perf_counter()-t0)/1000

# 배치 추론 (대규모 DOE 시나리오)
Xbig = torch.randn(100000, 3, device=device)
with torch.no_grad():
    model(Xbig[:1000])
    if torch.cuda.is_available(): torch.cuda.synchronize()
    t0 = time.perf_counter()
    model(Xbig)
    if torch.cuda.is_available(): torch.cuda.synchronize()
    T_INFER_BATCH = time.perf_counter()-t0

print("=== 속도 비교 ===")
print(f"FEM 솔버 1케이스           : {T_PER_SOLVE*1000:10.2f} ms")
print(f"서로게이트 1케이스          : {T_INFER_1*1e6:10.2f} us")
print(f"  -> 단일 케이스 가속       : {T_PER_SOLVE/T_INFER_1:10,.0f} 배")
print()
print(f"서로게이트 100,000건 배치   : {T_INFER_BATCH*1000:10.2f} ms")
print(f"동일 물량을 FEM으로 계산 시 : {T_PER_SOLVE*100000/3600:10.2f} 시간")
print(f"  -> 배치 처리 가속         : {T_PER_SOLVE*100000/T_INFER_BATCH:10,.0f} 배")
print()
print(f"[투자 회수 관점] 데이터셋 생성 {T_DATASET:.0f}s + 학습 {T_TRAIN:.0f}s = {T_DATASET+T_TRAIN:.0f}s 선투자")
print(f"  손익분기점: 약 {int((T_DATASET+T_TRAIN)/T_PER_SOLVE):,}건 이상 해석할 경우 서로게이트가 유리")

## 6. FP32 vs FP64 GPU 벤치마크 (L40S 특성 확인)

L40S가 왜 "솔버용"이 아니라 "서로게이트 학습용"인지를 수치로 확인합니다.
FEM 솔버의 핵심 연산인 밀집행렬 분해/곱을 FP32와 FP64로 각각 측정합니다.

In [ ]:
def bench(dtype, n=4096, reps=5):
    if not torch.cuda.is_available(): return None
    a = torch.randn(n, n, device=device, dtype=dtype)
    b = torch.randn(n, n, device=device, dtype=dtype)
    a @ b; torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(reps): a @ b
    torch.cuda.synchronize()
    dt = (time.perf_counter()-t0)/reps
    return 2*n**3/dt/1e12   # TFLOPS

if torch.cuda.is_available():
    # TF32를 끄고 순수 FP32 측정
    torch.backends.cuda.matmul.allow_tf32 = False
    fp32 = bench(torch.float32)
    torch.backends.cuda.matmul.allow_tf32 = True
    tf32 = bench(torch.float32)
    fp64 = bench(torch.float64)

    print(f"{GPU_NAME} 실측 (4096x4096 행렬곱)")
    print("-"*46)
    print(f"  FP32 (TF32 off) : {fp32:8.2f} TFLOPS")
    print(f"  TF32 (on)       : {tf32:8.2f} TFLOPS")
    print(f"  FP64            : {fp64:8.2f} TFLOPS")
    print()
    print(f"  FP32 / FP64 비율 : {fp32/fp64:.1f} 배")
    print()
    print("해석:")
    print("  · FP64가 FP32보다 현저히 느리면 -> 전통적 FEM/CFD 솔버 이식에 불리 (A100/H100이 유리)")
    print("  · FP32/TF32가 빠르면 -> 서로게이트 신경망 학습에 유리 (L40S의 적합 용도)")
    print("  · 즉 L40S는 '해석을 대신 푸는 GPU'가 아니라 '해석 결과를 학습하는 GPU'")
else:
    fp32 = tf32 = fp64 = None
    print("GPU 미할당 - 벤치마크 생략")

## 7. 결과 저장 (MyDisk)

In [ ]:
import os, json

# 마이디스크 실제 마운트 경로 탐색 (UI 표시명과 파일시스템 경로가 다름)
candidates = ["/root/kadap/MyDisk", "/root/자동차데이터플랫폼(KADaP)/MyDisk", "/workspace"]
base = next((p for p in candidates if os.path.isdir(p)), ".")
save_dir = os.path.join(base, "practice", "fem_surrogate")
os.makedirs(save_dir, exist_ok=True)
print("저장 위치:", save_dir)

# 1) 모델 가중치 + 정규화 파라미터 (재사용에 필수)
torch.save({
    "state_dict": model.state_dict(),
    "x_mu": x_mu, "x_sd": x_sd, "y_mu": y_mu, "y_sd": y_sd,
    "input_names": ["H_m", "t_m", "P_N"],
    "output_names": ["vonMises_Pa", "tip_deflection_m"],
    "fixed": {"L_m": L, "E_Pa": E, "nu": 0.3},
}, os.path.join(save_dir, "surrogate.pt"))

# 2) 해석 데이터셋
np.savez_compressed(os.path.join(save_dir, "fem_dataset.npz"), X=X_raw, Y=Y_raw)

# 3) 실습 결과 요약
summary = {
    "gpu": GPU_NAME,
    "n_samples": int(N_SAMPLES),
    "fem_solve_ms": round(T_PER_SOLVE*1000, 3),
    "dataset_gen_s": round(T_DATASET, 1),
    "train_s": round(T_TRAIN, 1),
    "surrogate_infer_us": round(T_INFER_1*1e6, 3),
    "speedup_single": round(T_PER_SOLVE/T_INFER_1, 1),
    "mean_err_stress_pct": round(float(err[:,0].mean()), 3),
    "mean_err_deflection_pct": round(float(err[:,1].mean()), 3),
    "tflops": {"fp32": fp32, "tf32": tf32, "fp64": fp64},
}
with open(os.path.join(save_dir, "result_summary.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print()
for fn in sorted(os.listdir(save_dir)):
    print(f"  {fn:24s} {os.path.getsize(os.path.join(save_dir, fn)):>12,} bytes")
print()
print(json.dumps(summary, ensure_ascii=False, indent=2))

## 8. 저장한 서로게이트 재사용 (배포 시나리오 확인)

학습된 서로게이트를 다시 불러와 사용하는 과정입니다.
실무에서는 이 단계가 "다른 사람/다른 워크로드가 학습 결과를 이어받아 쓰는" 방식에 해당하고,
스터디에서 정리한 **DGX(학습) → Jetson(배포)** 의 배포 측에 대응합니다.

In [ ]:
ckpt = torch.load(os.path.join(save_dir, "surrogate.pt"), map_location=device, weights_only=False)

m2 = nn.Sequential(nn.Linear(3,128), nn.SiLU(), nn.Linear(128,128), nn.SiLU(),
                   nn.Linear(128,64), nn.SiLU(), nn.Linear(64,2)).to(device)
m2.load_state_dict(ckpt["state_dict"]); m2.eval()

def predict(H_mm, t_mm, P_N):
    """설계변수를 넣으면 응력/처짐을 즉시 반환"""
    x = np.log(np.array([[H_mm/1000, t_mm/1000, P_N]]))
    xn = torch.tensor((x-ckpt["x_mu"])/ckpt["x_sd"], dtype=torch.float32, device=device)
    with torch.no_grad():
        y = m2(xn).cpu().numpy()
    out = np.exp(y*ckpt["y_sd"] + ckpt["y_mu"])[0]
    return out[0]/1e6, out[1]*1000   # MPa, mm

print("복원된 서로게이트로 즉시 예측 (FEM 대조)")
print(f"{'H[mm]':>7}{'t[mm]':>7}{'P[N]':>7} | {'응력_대체':>9}{'응력_FEM':>9} | {'처짐_대체':>9}{'처짐_FEM':>9}")
print("-"*70)
for H_mm, t_mm, P_N in [(80,10,2000), (120,15,3000), (160,25,4500)]:
    s_p, d_p = predict(H_mm, t_mm, P_N)
    vm, tp = solve_cantilever(L=L, H=H_mm/1000, t=t_mm/1000, E=E, P=P_N)
    print(f"{H_mm:7.0f}{t_mm:7.0f}{P_N:7.0f} | {s_p:9.2f}{vm/1e6:9.2f} | {d_p:9.4f}{tp*1000:9.4f}")

## 체크리스트

- [ ] 할당된 GPU가 **L40S**로 표시되고 VRAM이 44GB 내외로 잡혔다
- [ ] FEM 결과가 보 이론과 수 % 내로 일치했다 (솔버 타당성)
- [ ] 데이터셋 1,000건 생성 시간을 기록했다
- [ ] 서로게이트 평균 예측 오차가 수 % 이내로 수렴했다
- [ ] 단일 케이스 추론이 FEM보다 수천~수만 배 빠른 것을 확인했다
- [ ] FP64가 FP32보다 느린 것을 확인했다 (L40S 특성)
- [ ] 결과 파일 3종이 MyDisk에 저장됐다
- [ ] 저장한 서로게이트를 불러와 재예측이 되는 것을 확인했다

## 정리 — 이 실습이 보여주는 것

**1. L40S의 적합 용도**

FP64가 느리므로 상용 FEM/CFD 솔버를 GPU로 옮기는 용도로는 부적합하고, FP32/TF32가 빠르므로 해석 결과를 학습하는 서로게이트 모델 학습에 적합합니다. 스터디에서 정리한 "DGX = 서로게이트를 만드는 단계"가 CAE 업무에서는 정확히 이 형태로 나타납니다.

**2. 서로게이트의 손익분기점**

선투자(데이터셋 생성 + 학습)를 회수하려면 일정 건수 이상의 해석이 필요합니다. 단발성 해석에는 의미가 없고, 최적설계·민감도분석·실시간 응답처럼 **같은 해석을 반복 호출하는 업무**에서 가치가 생깁니다.

**3. 실물 CAE로 확장할 때 달라지는 점**

이 실습의 FEM은 자유도 수백 개 수준이라 1케이스가 수십 ms입니다. 실제 차량 충돌·NVH 해석은 수백만 자유도에 1케이스가 수 시간이므로, 데이터셋 생성 비용이 압도적으로 커집니다. 따라서 실무 적용 시에는 샘플 수를 줄이면서 정확도를 확보하는 설계(적응적 샘플링, 전이학습, 물리정보신경망 등)가 핵심 과제가 됩니다.

**4. 다음 단계 후보**

- 설계변수를 늘려(재질, 형상 파라미터) 고차원 서로게이트의 한계 확인
- Batch Job으로 전환해 데이터셋 생성을 병렬 분산 처리
- 동일 실습을 A100에서 돌려 FP64 성능 차이를 실측 비교